# Polars: The Lightning-Fast DataFrame Library

## What Is Polars?

Imagine you need to sort 1 million books alphabetically.  
Pandas uses one librarian working sequentially — takes an hour.  
**Polars** uses 8 librarians working in parallel, each sorting a section simultaneously — done in 10 minutes.

**Polars** is a DataFrame library written in **Rust** with Python bindings.  
It's designed from scratch for speed — not a wrapper around NumPy like pandas.

Key advantages:
- **Parallel by default**: uses all CPU cores automatically
- **Lazy evaluation**: builds a query plan, optimizes it, then executes
- **Memory efficient**: uses Apache Arrow columnar format
- **No index**: unlike pandas, no confusing row index — just columns
- **Expressive API**: method chaining with `pl.col()` expressions

## Resources

- **Docs**: [https://docs.pola.rs/](https://docs.pola.rs/)
- **User Guide**: [https://docs.pola.rs/user-guide/](https://docs.pola.rs/user-guide/)
- **GitHub**: [https://github.com/pola-rs/polars](https://github.com/pola-rs/polars)
- **YouTube — Polars Tutorial**: [https://www.youtube.com/watch?v=VHqn7ufiilE](https://www.youtube.com/watch?v=VHqn7ufiilE)
- **Polars vs Pandas benchmarks**: [https://www.pola.rs/benchmarks.html](https://www.pola.rs/benchmarks.html)

## Installation

```bash
pip install polars
# With all optional extras (Excel, JSON, cloud storage)
pip install 'polars[all]'
```

In [ ]:
import numpy as np
import time

try:
    import polars as pl
    POLARS_AVAILABLE = True
    print(f"Polars version: {pl.__version__}")
except ImportError:
    POLARS_AVAILABLE = False
    print("Polars not installed — simulated output shown. Install: pip install polars")

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
    print(f"Pandas version: {pd.__version__}")
except ImportError:
    PANDAS_AVAILABLE = False

# Create synthetic sales dataset (500K rows)
np.random.seed(42)
N = 500_000

regions    = np.random.choice(['North', 'South', 'East', 'West'], N)
categories = np.random.choice(['Electronics', 'Clothing', 'Food', 'Books', 'Sports'], N)
sales      = np.round(np.random.exponential(scale=200, size=N), 2)
quantities = np.random.randint(1, 50, N)

print(f"\nSynthetic dataset ready: {N:,} rows")
print(f"Columns: region, category, sales, quantity")

## Core Concept 1: Creating DataFrames

Polars DataFrames are created from Python dicts, lists, or converted from pandas.  
Key difference from pandas: **no row index** — rows are just rows, numbered 0, 1, 2, ...

In [ ]:
if POLARS_AVAILABLE:
    # Create Polars DataFrame from dict
    df = pl.DataFrame({
        'region':   regions,
        'category': categories,
        'sales':    sales,
        'quantity': quantities,
    })

    print("DataFrame info:")
    print(f"  Shape:   {df.shape}")
    print(f"  Columns: {df.columns}")
    print(f"  Dtypes:  {df.dtypes}")
    print()
    print("First 5 rows:")
    print(df.head())
    print()
    print("Schema (column → dtype):")
    print(df.schema)
    print()
    print("Descriptive statistics:")
    print(df.describe())

    # Convert from / to pandas
    if PANDAS_AVAILABLE:
        pd_df = pd.DataFrame({'a': [1, 2, 3], 'b': ['x', 'y', 'z']})
        pl_from_pd = pl.from_pandas(pd_df)          # pandas → polars
        back_to_pd = pl_from_pd.to_pandas()         # polars → pandas
        print(f"\npandas → polars → pandas round-trip: {type(back_to_pd).__name__}")

else:
    print("Creating Polars DataFrame (simulated):")
    print()
    print("  df = pl.DataFrame({")
    print("      'region':   regions,    # numpy array or list")
    print("      'category': categories,")
    print("      'sales':    sales,")
    print("      'quantity': quantities,")
    print("  })")
    print()
    print("  shape: (500000, 4)")
    print("  dtypes: [String, String, Float64, Int64]")
    print()
    print("  ┌────────┬─────────────┬────────┬──────────┐")
    print("  │ region ┆ category    ┆ sales  ┆ quantity │")
    print("  │ ---    ┆ ---         ┆ ---    ┆ ---      │")
    print("  │ str    ┆ str         ┆ f64    ┆ i64      │")
    print("  ╞════════╪═════════════╪════════╪══════════╡")
    print("  │ North  ┆ Electronics ┆ 182.34 ┆ 12       │")
    print("  │ South  ┆ Clothing    ┆  45.67 ┆  3       │")
    print("  └────────┴─────────────┴────────┴──────────┘")

## Core Concept 2: Selecting, Filtering, and Adding Columns

Polars uses `pl.col()` expressions — more explicit than pandas string column names.  
This enables powerful query optimization and parallel execution.

In [ ]:
if POLARS_AVAILABLE:
    # SELECT: choose columns
    print("1. Select specific columns:")
    print(df.select(['region', 'sales']).head(3))
    print()

    # SELECT with computed columns
    print("2. Select with a computed column:")
    result = df.select([
        pl.col('region'),
        pl.col('sales'),
        (pl.col('sales') * pl.col('quantity')).alias('revenue'),
    ])
    print(result.head(3))
    print()

    # FILTER: like SQL WHERE
    print("3. Filter: sales > 500 AND region == 'North'")
    high_north = df.filter(
        (pl.col('sales') > 500) & (pl.col('region') == 'North')
    )
    print(f"   Rows: {len(high_north):,} / {len(df):,}")
    print(high_north.head(3))
    print()

    # WITH_COLUMNS: add or modify columns (like pandas .assign())
    print("4. Add multiple columns with with_columns:")
    df2 = df.with_columns([
        (pl.col('sales') * pl.col('quantity')).alias('revenue'),
        pl.col('sales').log(base=10).alias('log_sales'),
        (pl.col('region') + '_' + pl.col('category')).alias('segment'),
    ])
    print(df2.head(3))

else:
    print("Selection, filtering, adding columns (simulated):")
    print()
    print("  # Select columns")
    print("  df.select(['region', 'sales'])")
    print()
    print("  # Computed column in select")
    print("  df.select([")
    print("      pl.col('region'),")
    print("      (pl.col('sales') * pl.col('quantity')).alias('revenue'),")
    print("  ])")
    print()
    print("  # Filter (like SQL WHERE)")
    print("  df.filter((pl.col('sales') > 500) & (pl.col('region') == 'North'))")
    print("  → 9,876 rows")
    print()
    print("  # Add columns (like pandas .assign())")
    print("  df.with_columns([")
    print("      (pl.col('sales') * pl.col('quantity')).alias('revenue'),")
    print("      pl.col('sales').log(base=10).alias('log_sales'),")
    print("  ])")

## Core Concept 3: GroupBy and Aggregations

GroupBy in Polars runs across all CPU cores in parallel.  
Multiple aggregations on the same group_by execute simultaneously.

In [ ]:
if POLARS_AVAILABLE:
    print("GroupBy with multiple aggregations:")
    summary = (
        df.group_by(['region', 'category'])
        .agg([
            pl.col('sales').sum().alias('total_sales'),
            pl.col('sales').mean().round(2).alias('avg_sales'),
            pl.col('sales').max().alias('max_sales'),
            pl.col('quantity').sum().alias('total_qty'),
            pl.len().alias('num_transactions'),
        ])
        .sort('total_sales', descending=True)
    )
    print(summary.head(10))
    print()

    # Benchmark: Polars vs pandas
    if PANDAS_AVAILABLE:
        pd_df = df.to_pandas()

        t0 = time.time()
        for _ in range(5):
            pd_df.groupby(['region', 'category'])['sales'].agg(['sum', 'mean', 'max'])
        pd_time = (time.time() - t0) / 5

        t0 = time.time()
        for _ in range(5):
            df.group_by(['region', 'category']).agg([
                pl.col('sales').sum(), pl.col('sales').mean(), pl.col('sales').max()
            ])
        pl_time = (time.time() - t0) / 5

        print(f"GroupBy benchmark (500K rows, 5-run average):")
        print(f"  Pandas: {pd_time*1000:.1f} ms")
        print(f"  Polars: {pl_time*1000:.1f} ms")
        print(f"  Speedup: {pd_time/pl_time:.1f}×")

else:
    print("GroupBy (simulated):")
    print()
    print("  df.group_by(['region', 'category'])")
    print("  .agg([")
    print("      pl.col('sales').sum().alias('total_sales'),")
    print("      pl.col('sales').mean().round(2).alias('avg_sales'),")
    print("      pl.len().alias('num_transactions'),")
    print("  ])")
    print("  .sort('total_sales', descending=True)")
    print()
    print("  GroupBy benchmark (500K rows):")
    print("    Pandas:  310 ms")
    print("    Polars:   27 ms")
    print("    Speedup: 11.5×")

## Core Concept 4: Lazy Mode — The Secret Weapon

**Lazy mode** = describe what you want, let Polars figure out the best way to do it.

Think of it like ordering food at a restaurant vs cooking it yourself:  
You describe the meal you want, and the chef (Polars optimizer) decides the most efficient way to make it.

Polars' **query optimizer** can:
- **Predicate pushdown**: filter rows as early as possible (read less data)
- **Projection pushdown**: drop unused columns early
- **Reorder operations**: for maximum efficiency
- **Parallelize**: independent operations run simultaneously

In [ ]:
if POLARS_AVAILABLE:
    # Convert to LazyFrame — nothing runs yet
    lf = df.lazy()

    # Build a complex query chain — still nothing runs!
    query = (
        lf
        .filter(pl.col('sales') > 100)                           # step 1
        .with_columns(
            (pl.col('sales') * pl.col('quantity')).alias('revenue')
        )                                                         # step 2
        .group_by('region')                                       # step 3
        .agg([
            pl.col('revenue').sum().alias('total_revenue'),
            pl.col('sales').mean().round(2).alias('avg_sales'),
            pl.len().alias('num_orders'),
        ])
        .sort('total_revenue', descending=True)
        .limit(10)
    )

    print("Optimized query plan:")
    try:
        print(query.explain())  # shows what Polars will actually do
    except Exception:
        print("  [use query.explain() to see the optimized plan]")
    print()

    # .collect() is the ONLY place computation happens
    t0 = time.time()
    result = query.collect()
    print(f"Result (computed in {(time.time()-t0)*1000:.1f} ms):")
    print(result)

else:
    print("Lazy mode (simulated):")
    print()
    print("  lf = df.lazy()           # → LazyFrame (no computation)")
    print()
    print("  query = (")
    print("      lf")
    print("      .filter(pl.col('sales') > 100)   # added to plan")
    print("      .with_columns([...])              # added to plan")
    print("      .group_by('region').agg([...])    # added to plan")
    print("      .sort('total_revenue', descending=True)")
    print("  )")
    print()
    print("  result = query.collect()   # NOW the optimizer runs and executes")
    print()
    print("  Simulated result:")
    print("  ┌────────┬───────────────┬───────────┬────────────┐")
    print("  │ region ┆ total_revenue ┆ avg_sales ┆ num_orders │")
    print("  ╞════════╪═══════════════╪═══════════╪════════════╡")
    print("  │ West   ┆ 6,234,567.89  ┆ 198.34    ┆ 62,456     │")
    print("  │ North  ┆ 6,198,234.12  ┆ 197.82    ┆ 62,123     │")
    print("  └────────┴───────────────┴───────────┴────────────┘")

## Core Concept 5: File I/O — Lazy Scanning

Polars can read CSV, Parquet, JSON, and more.  
**`scan_csv`** / **`scan_parquet`** reads lazily — only loads what the query actually needs.

In [ ]:
import os, tempfile, shutil

if POLARS_AVAILABLE:
    tmp_dir      = tempfile.mkdtemp()
    csv_path     = os.path.join(tmp_dir, 'sales.csv')
    parquet_path = os.path.join(tmp_dir, 'sales.parquet')

    df.write_csv(csv_path)
    df.write_parquet(parquet_path)

    csv_size = os.path.getsize(csv_path) / 1e6
    pq_size  = os.path.getsize(parquet_path) / 1e6
    print(f"Written CSV:     {csv_size:.1f} MB")
    print(f"Written Parquet: {pq_size:.1f} MB  ({csv_size/pq_size:.1f}× smaller!)")
    print()

    # scan_csv: lazy — Polars pushes the filter down into the reader
    t0 = time.time()
    result_csv = (
        pl.scan_csv(csv_path)
        .filter(pl.col('region') == 'North')
        .select(['sales', 'quantity'])
        .mean()
        .collect()
    )
    csv_time = time.time() - t0

    # scan_parquet: faster (columnar + compressed — reads only needed columns)
    t0 = time.time()
    result_pq = (
        pl.scan_parquet(parquet_path)
        .filter(pl.col('region') == 'North')
        .select(['sales', 'quantity'])
        .mean()
        .collect()
    )
    parquet_time = time.time() - t0

    print("Mean sales and quantity for North region:")
    print(result_csv)
    print(f"\nScan CSV:     {csv_time*1000:.1f} ms")
    print(f"Scan Parquet: {parquet_time*1000:.1f} ms")
    print(f"Parquet speedup: {csv_time/parquet_time:.1f}×")

    shutil.rmtree(tmp_dir)

else:
    print("File I/O (simulated):")
    print()
    print("  # Write")
    print("  df.write_csv('sales.csv')          # → 18 MB")
    print("  df.write_parquet('sales.parquet')   # →  4 MB (4.5× smaller!)")
    print()
    print("  # Lazy scan — optimizer pushes filter into reader")
    print("  pl.scan_csv('sales.csv')")
    print("      .filter(pl.col('region') == 'North')")
    print("      .select(['sales', 'quantity'])")
    print("      .mean()")
    print("      .collect()")
    print()
    print("  Scan CSV:     240 ms")
    print("  Scan Parquet:  35 ms  (7× faster)")

## Core Concept 6: Window Functions with `.over()`

`.over()` computes per-group aggregations while keeping all original rows — like SQL `PARTITION BY`.

In [ ]:
if POLARS_AVAILABLE:
    # Window functions: compute per group but keep all rows
    df_window = df.head(20).with_columns([
        # Average sales for this row's region (same value for all rows in region)
        pl.col('sales').mean().over('region').alias('region_avg'),

        # Rank within region (1 = highest sales in that region)
        pl.col('sales').rank(method='ordinal', descending=True).over('region').alias('rank_in_region'),

        # Running total within category
        pl.col('sales').cum_sum().over('category').alias('cumulative_sales'),
    ])

    print("Window functions with .over():")
    print(df_window.select(['region', 'category', 'sales', 'region_avg', 'rank_in_region', 'cumulative_sales']))

else:
    print("Window functions (simulated):")
    print()
    print("  df.with_columns([")
    print("      # Mean for each row's region (keeps all rows)")
    print("      pl.col('sales').mean().over('region').alias('region_avg'),")
    print()
    print("      # Rank within region (1 = highest sales)")
    print("      pl.col('sales').rank(descending=True).over('region').alias('rank_in_region'),")
    print()
    print("      # Running total within category")
    print("      pl.col('sales').cum_sum().over('category').alias('cumulative_sales'),")
    print("  ])")
    print()
    print("  SQL equivalent:")
    print("    AVG(sales) OVER (PARTITION BY region) AS region_avg")
    print("    RANK() OVER (PARTITION BY region ORDER BY sales DESC)")

## Core Concept 7: String and DateTime Operations

Polars has specialized namespaces for string (`.str.*`) and datetime (`.dt.*`) operations.

In [ ]:
if POLARS_AVAILABLE:
    import datetime

    # String operations
    df_str = pl.DataFrame({
        'name':  ['Alice Smith', 'Bob Jones', 'Carol White', 'Dave Brown'],
        'email': ['alice@example.com', 'bob@test.org', 'carol@work.net', 'dave@example.com'],
        'code':  ['A001', 'B002', 'C003', 'D004'],
    })

    print("String operations (.str namespace):")
    result_str = df_str.with_columns([
        pl.col('name').str.to_uppercase().alias('name_upper'),
        pl.col('name').str.split(' ').list.first().alias('first_name'),
        pl.col('email').str.contains('@example.com').alias('is_example'),
        pl.col('code').str.slice(1).cast(pl.Int32).alias('code_num'),
    ])
    print(result_str)
    print()

    # DateTime operations
    df_dt = pl.DataFrame({
        'date': pl.date_range(
            start=datetime.date(2023, 1, 1),
            end=datetime.date(2023, 12, 31),
            interval='1mo',
            eager=True
        ),
        'value': [100, 120, 95, 140, 110, 130, 155, 148, 162, 170, 145, 190],
    })

    print("DateTime operations (.dt namespace):")
    result_dt = df_dt.with_columns([
        pl.col('date').dt.month().alias('month'),
        pl.col('date').dt.quarter().alias('quarter'),
        pl.col('date').dt.strftime('%B %Y').alias('month_name'),
    ])
    print(result_dt)

else:
    print("String and DateTime (simulated):")
    print()
    print("  # String ops")
    print("  pl.col('name').str.to_uppercase()")
    print("  pl.col('name').str.split(' ').list.first()   # first word")
    print("  pl.col('email').str.contains('@example.com') # boolean")
    print("  pl.col('code').str.slice(1).cast(pl.Int32)   # 'A001' → 1")
    print()
    print("  # DateTime ops")
    print("  pl.col('date').dt.month()")
    print("  pl.col('date').dt.quarter()")
    print("  pl.col('date').dt.strftime('%B %Y')  # → 'January 2023'")

## Common Pitfalls

| Pitfall | Symptom | Fix |
|---------|---------|-----|
| Calling `.collect()` inside a loop | Re-executes query each iteration | Build whole lazy query first, then one `.collect()` |
| Using pandas-style `df['col']` for computation | Works but loses lazy benefit | Use `pl.col('col')` in `.select()` / `.with_columns()` |
| Forgetting `.alias()` | Column named `literal` or `sales` (same as source) | Always `.alias('new_name')` after computed expressions |
| `.to_pandas()` early then pandas ops | Kills all speedup | Stay in Polars; only convert at the very end if needed |
| Confusing `null` vs `NaN` | Wrong null counts | `null` = missing; `NaN` = float math error. Use `.is_null()` vs `.is_nan()` |
| `sort` without specifying direction | Ascending by default | Use `descending=True` explicitly |

## Mini Project: Full Sales Analysis Pipeline

In [ ]:
if POLARS_AVAILABLE:
    print("=" * 60)
    print("POLARS SALES ANALYSIS PIPELINE")
    print("=" * 60)
    print()

    # Build a full lazy pipeline
    t0 = time.time()

    report = (
        df.lazy()

        # 1. Remove zero/negative sales
        .filter(pl.col('sales') > 0)

        # 2. Add revenue and normalize sales
        .with_columns([
            (pl.col('sales') * pl.col('quantity')).alias('revenue'),
            pl.col('region').str.to_uppercase().alias('REGION'),
        ])

        # 3. Aggregate by region and category
        .group_by(['REGION', 'category'])
        .agg([
            pl.col('revenue').sum().alias('total_revenue'),
            pl.col('sales').mean().round(2).alias('avg_sale_price'),
            pl.col('quantity').sum().alias('units_sold'),
            pl.len().alias('transactions'),
        ])

        # 4. Add per-region rank (window function)
        .with_columns(
            pl.col('total_revenue')
              .rank(method='ordinal', descending=True)
              .over('REGION')
              .alias('rank_in_region')
        )

        # 5. Keep only top 2 per region
        .filter(pl.col('rank_in_region') <= 2)
        .sort(['REGION', 'rank_in_region'])
        .collect()
    )

    elapsed = time.time() - t0
    print(f"Computed in {elapsed*1000:.1f} ms for {N:,} input rows")
    print()
    print("Top 2 revenue categories per region:")
    print(report.to_string())
    print()
    print(f"Grand total revenue: ${report['total_revenue'].sum():,.0f}")

else:
    print("Mini Project: Sales Pipeline (simulated output)")
    print()
    print("  Computed in 42 ms for 500,000 input rows")
    print()
    print("  REGION  category     total_revenue  avg_price  units  transactions  rank")
    print("  EAST    Electronics  1,823,456      198.4      9,187  62,456        1")
    print("  EAST    Sports       1,756,234      193.2      9,089  58,234        2")
    print("  NORTH   Electronics  1,801,123      197.8      9,100  61,234        1")
    print("  NORTH   Books        1,743,678      191.3      9,123  59,456        2")
    print("  SOUTH   Food         1,799,234      197.2      9,145  60,789        1")
    print("  SOUTH   Clothing     1,731,456      190.1      9,078  57,891        2")
    print("  WEST    Sports       1,812,345      198.9      9,112  61,892        1")
    print("  WEST    Electronics  1,798,123      197.1      9,133  60,567        2")
    print()
    print("  Grand total revenue: $14,265,649")

## Interview Questions and Answers

In [ ]:
qa = [
    {"q": "Why is Polars faster than pandas?",
     "a": """Polars is faster for several architectural reasons:

1. Written in Rust: zero garbage collector, no Python overhead per row.
   pandas is Python + NumPy (C) but with Python glue that adds overhead.

2. Parallel by default: Polars uses Rayon (Rust parallel lib) — all CPU cores
   work simultaneously. pandas is single-threaded for most operations.

3. Apache Arrow columnar memory format:
   Data stored column-by-column. Aggregations (sum, mean) read one column
   → perfect CPU cache behavior (sequential memory access).

4. Query optimizer (lazy mode):
   - Predicate pushdown: filters applied as early as possible
   - Projection pushdown: unused columns dropped before computation
   - Similar to how SQL databases optimize queries.

5. No Python for-loops: everything stays in Rust.
   pandas' .apply() (Python function per row) is 100-1000× slower.

Typical speedups: 5-30× for groupby/filter; 50-100× for complex pipelines."""},

    {"q": "What is lazy evaluation and what are its benefits?",
     "a": """Lazy evaluation means 'describe the computation first, execute it later.'

How:
  df.lazy()       → creates a LazyFrame (computation graph)
  .filter(...)    → adds a node to the graph
  .group_by(...)  → adds another node
  .collect()      → optimizer runs, then executes the whole graph

Benefits:

Predicate pushdown:
  Eager:  read 500K rows → filter to 10K → aggregate
  Lazy:   (optimizer) filter during read → aggregate 10K rows only
  → reads 50× less data from disk

Projection pushdown:
  Eager:  read 20 columns → select 2 → compute
  Lazy:   read only 2 columns → compute
  → 10× less memory used

Common subplan elimination:
  If a subquery appears twice, compute it once and reuse.

Key API:
  scan_csv()    = lazy file read (use this for large files)
  read_csv()    = eager file read (loads everything)
  .collect()    = triggers actual computation"""},

    {"q": "Polars vs Dask vs Spark — when would you choose each?",
     "a": """Choose based on data size and infrastructure:

Polars (single machine, fast):
  - Data fits in RAM of one machine
  - Up to ~100GB on a beefy machine (512GB RAM)
  - No cluster, no configuration, just install and run
  - Best performance per machine

Dask (distributed Python):
  - Data exceeds single machine RAM
  - Familiar pandas-like API
  - Works on local multi-core or multi-machine clusters
  - Slower per operation (coordination overhead)

PySpark (enterprise big data):
  - Petabyte scale datasets
  - Existing Hadoop/YARN/Databricks cluster
  - Strong ecosystem (Delta Lake, MLlib)
  - High latency per job (JVM startup)

Rule of thumb:
  < 10GB      → pandas (simplest)
  10-100GB    → Polars (fastest, single machine)
  > 100GB     → Dask or Spark (distributed)

Modern trend: Polars + DuckDB replacing Dask for many 10-100GB use cases."""},

    {"q": "What is the .over() function and when would you use it?",
     "a": """`.over()` is Polars' window function — SQL's PARTITION BY equivalent.

It computes a per-group aggregation while KEEPING all original rows.
group_by reduces rows; .over() does not.

Syntax:
  pl.col('sales').mean().over('region')  # mean per region, for every row

SQL equivalent:
  AVG(sales) OVER (PARTITION BY region) AS region_avg

Common use cases:
  # Add group average for comparison
  (pl.col('sales') - pl.col('sales').mean().over('region')).alias('sales_vs_region_avg')

  # Rank within group
  pl.col('sales').rank(descending=True).over('region').alias('rank_in_region')

  # Running total within group
  pl.col('sales').cum_sum().over('category').alias('running_total')

  # Percent of group total
  (pl.col('sales') / pl.col('sales').sum().over('region')).alias('share_of_region')

Polars runs .over() computations in parallel for each partition."""},

    {"q": "How do you handle missing values in Polars?",
     "a": """Polars distinguishes between null (missing) and NaN (not-a-number).

Detecting:
  df.null_count()                          # count nulls per column
  df.filter(pl.col('col').is_null())       # rows where null
  df.filter(pl.col('col').is_not_null())   # rows where not null

Filling nulls:
  df.fill_null(0)                          # fill all nulls with 0
  df.fill_null(strategy='forward')         # propagate last valid value
  df.fill_null(strategy='backward')        # propagate next valid value
  df.with_columns(
      pl.col('price').fill_null(pl.col('price').mean())
  )                                        # fill with column mean

Dropping nulls:
  df.drop_nulls()                          # drop rows with any null
  df.drop_nulls(subset=['col1', 'col2'])   # only check these columns

NaN (float errors like 0/0):
  df.filter(pl.col('val').is_nan())        # find NaN rows
  df.with_columns(pl.col('val').fill_nan(0)) # replace NaN with 0"""},

    {"q": "How would you use Polars in an ML preprocessing pipeline?",
     "a": """Polars replaces pandas in the preprocessing step of ML pipelines.

1. One-hot encoding:
   df.to_dummies(columns=['region', 'category'])
   # → adds region_North, region_South, category_Electronics, ...

2. Normalization:
   df.with_columns(
       ((pl.col('sales') - pl.col('sales').mean()) /
        pl.col('sales').std()).alias('sales_z')
   )

3. Time series lag features:
   df.with_columns([
       pl.col('value').shift(1).alias('lag_1'),
       pl.col('value').shift(7).alias('lag_7'),
       pl.col('value').rolling_mean(window_size=7).alias('rolling_7d'),
   ])

4. Cross features:
   df.with_columns(
       (pl.col('region') + '_' + pl.col('category')).alias('segment')
   )

5. Train/test split (no sklearn needed for split):
   n = len(df)
   train = df.slice(0, int(0.8 * n))
   test  = df.slice(int(0.8 * n))

6. Convert to numpy for sklearn:
   X = df.select(feature_cols).to_numpy()
   y = df['label'].to_numpy()
   # Then use with sklearn, PyTorch, etc."""},
]

for i, item in enumerate(qa, 1):
    print(f"Q{i}: {item['q']}")
    print(f"A:  {item['a'].strip()}")
    print("-" * 65)
    print()

## Summary

| Operation | Polars API |
|-----------|------------|
| Create DataFrame | `pl.DataFrame({'col': data})` |
| Select columns | `df.select(['col1', 'col2'])` |
| Computed column | `df.select((pl.col('a') * pl.col('b')).alias('c'))` |
| Filter rows | `df.filter((pl.col('x') > 0) & (pl.col('y') == 'A'))` |
| Add column | `df.with_columns(expr.alias('name'))` |
| GroupBy | `df.group_by('col').agg([pl.col('val').sum()])` |
| Sort | `df.sort('col', descending=True)` |
| Join | `df.join(other, on='key', how='inner')` |
| Window function | `pl.col('val').mean().over('group')` |
| Lazy mode | `df.lazy()` → chain ops → `.collect()` |
| String ops | `pl.col('s').str.to_uppercase()` |
| DateTime ops | `pl.col('d').dt.month()` |
| Lazy file scan | `pl.scan_csv('file.csv').filter(...).collect()` |
| Write Parquet | `df.write_parquet('file.parquet')` |
| To numpy | `df.select(cols).to_numpy()` |
| Null handling | `df.fill_null(0)` / `df.drop_nulls()` |

### Next Steps
1. **Polars user guide**: [https://docs.pola.rs/user-guide/](https://docs.pola.rs/user-guide/)
2. **Polars vs pandas migration**: [https://docs.pola.rs/user-guide/migration/pandas/](https://docs.pola.rs/user-guide/migration/pandas/)
3. **Next**: PySpark for distributed data processing at petabyte scale